# 01 — ETL Pipeline · H&M In-Store Movement & Product Placement

**Session 2 Deliverable** — DuckDB load, clean, and export

| Step | What happens |
|------|--------------|
| 1 | Load raw H&M CSV files into DuckDB |
| 2 | Inspect schema and row counts |
| 3 | Clean with COALESCE / CASE WHEN / CAST |
| 4 | Export to Parquet (analytical side) |
| 5 | Export Neo4j-ready CSVs (one per node label, one per relationship type) |

**Dataset files expected in** `data/raw/`:
- `articles.csv` — 105,542 products with descriptions, colours, categories
- `customers.csv` — 1,371,980 customers with age, membership, activity
- `transactions_train.csv` — 31,788,324 purchase transactions

> **Scope for tonight:** Load all three files, clean, and export node/edge CSVs for
> `Article`, `Customer`, and `PURCHASED` relationship. Full schema is Session 3.

In [46]:
# ── 0. Install / import dependencies ─────────────────────────────────────────
# Run once if not already installed:
!pip install duckdb pandas pyarrow

import duckdb
import pandas as pd
import os
from pathlib import Path

print(f"DuckDB version : {duckdb.__version__}")
print(f"Pandas version : {pd.__version__}")

DuckDB version : 1.5.2
Pandas version : 2.2.3


In [47]:
from pathlib import Path

folder = Path(r'C:\Users\prasa\Downloads\H&M')

print(f"Contents of {folder}:\n")
for item in sorted(folder.rglob('*')):
    size_mb = item.stat().st_size / 1e6 if item.is_file() else 0
    print(f"  {'FILE' if item.is_file() else 'DIR ':4s}  {size_mb:8.1f} MB  {item.relative_to(folder)}")

Contents of C:\Users\prasa\Downloads\H&M:

  FILE      36.1 MB  articles.csv
  FILE     207.1 MB  customers.csv
  DIR        0.0 MB  neo4j
  DIR        0.0 MB  parquet
  FILE       5.7 MB  parquet\articles.parquet
  FILE     168.3 MB  parquet\customers.parquet
  FILE     807.7 MB  parquet\transactions.parquet
  FILE    3488.0 MB  transactions_train.csv


In [48]:
RAW_DIR     = Path('C:/Users/prasa/Downloads/H&M')
PARQUET_DIR = Path('C:/Users/prasa/Downloads/H&M/parquet')
NEO4J_DIR   = Path('C:/Users/prasa/Downloads/H&M/neo4j')

for d in [PARQUET_DIR, NEO4J_DIR]:
    d.mkdir(parents=True, exist_ok=True)

required = ['articles.csv', 'customers.csv', 'transactions_train.csv']
for f in required:
    path = RAW_DIR / f
    exists = path.exists()
    size   = f"{path.stat().st_size / 1e6:.1f} MB" if exists else "NOT FOUND"
    print(f"  {'✅' if exists else '❌'}  {f:35s}  {size}")

  ✅  articles.csv                         36.1 MB
  ✅  customers.csv                        207.1 MB
  ✅  transactions_train.csv               3488.0 MB


In [49]:
# ── 2. Connect DuckDB (in-memory) and load raw CSVs ──────────────────────────
con = duckdb.connect()  # in-memory; no file lock

# Register all three raw CSVs as views — DuckDB reads lazily
con.execute("""
    CREATE OR REPLACE VIEW raw_articles AS
    SELECT * FROM read_csv_auto('C:/Users/prasa/Downloads/H&M/articles.csv', header=true)
""")

con.execute("""
    CREATE OR REPLACE VIEW raw_customers AS
    SELECT * FROM read_csv_auto('C:/Users/prasa/Downloads/H&M/customers.csv', header=true)
""")

con.execute("""
    CREATE OR REPLACE VIEW raw_transactions AS
    SELECT * FROM read_csv_auto('C:/Users/prasa/Downloads/H&M/transactions_train.csv', header=true)
""")

print("Views registered. Counting rows...")
for view in ['raw_articles', 'raw_customers', 'raw_transactions']:
    n = con.execute(f"SELECT COUNT(*) FROM {view}").fetchone()[0]
    print(f"  {view:25s}  {n:>12,} rows")

Views registered. Counting rows...
  raw_articles                    105,542 rows
  raw_customers                 1,371,980 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  raw_transactions             31,788,324 rows


In [50]:
# ── 3. Inspect raw schemas ────────────────────────────────────────────────────
for view in ['raw_articles', 'raw_customers', 'raw_transactions']:
    print(f"\n{'─'*55}")
    print(f"  {view.upper()}")
    print(f"{'─'*55}")
    schema = con.execute(f"DESCRIBE {view}").df()
    print(schema[['column_name','column_type']].to_string(index=False))
    print()
    sample = con.execute(f"SELECT * FROM {view} LIMIT 3").df()
    print(sample.to_string())


───────────────────────────────────────────────────────
  RAW_ARTICLES
───────────────────────────────────────────────────────
                 column_name column_type
                  article_id     VARCHAR
                product_code     VARCHAR
                   prod_name     VARCHAR
             product_type_no      BIGINT
           product_type_name     VARCHAR
          product_group_name     VARCHAR
     graphical_appearance_no      BIGINT
   graphical_appearance_name     VARCHAR
           colour_group_code     VARCHAR
           colour_group_name     VARCHAR
   perceived_colour_value_id      BIGINT
 perceived_colour_value_name     VARCHAR
  perceived_colour_master_id      BIGINT
perceived_colour_master_name     VARCHAR
               department_no      BIGINT
             department_name     VARCHAR
                  index_code     VARCHAR
                  index_name     VARCHAR
              index_group_no      BIGINT
            index_group_name     VARCHAR
           

In [51]:
# ── 4a. Null / quality check ──────────────────────────────────────────────────
print("=== ARTICLES — null counts ===")
con.execute("""
    SELECT
        COUNT(*)                               AS total_rows,
        COUNT(*) - COUNT(article_id)           AS null_article_id,
        COUNT(*) - COUNT(product_code)         AS null_product_code,
        COUNT(*) - COUNT(prod_name)            AS null_prod_name,
        COUNT(*) - COUNT(colour_group_name)    AS null_colour,
        COUNT(*) - COUNT(detail_desc)          AS null_desc
    FROM raw_articles
""").df().T.rename(columns={0: 'count'}).pipe(print)

print("\n=== CUSTOMERS — null counts ===")
con.execute("""
    SELECT
        COUNT(*)                               AS total_rows,
        COUNT(*) - COUNT(customer_id)          AS null_customer_id,
        COUNT(*) - COUNT(age)                  AS null_age,
        COUNT(*) - COUNT(FN)                   AS null_fn,
        COUNT(*) - COUNT(Active)               AS null_active,
        COUNT(*) - COUNT(club_member_status)   AS null_member_status,
        COUNT(*) - COUNT(fashion_news_frequency) AS null_news_freq
    FROM raw_customers
""").df().T.rename(columns={0: 'count'}).pipe(print)

print("\n=== TRANSACTIONS — null counts ===")
con.execute("""
    SELECT
        COUNT(*)                               AS total_rows,
        COUNT(*) - COUNT(t_dat)                AS null_date,
        COUNT(*) - COUNT(customer_id)          AS null_customer_id,
        COUNT(*) - COUNT(article_id)           AS null_article_id,
        COUNT(*) - COUNT(price)                AS null_price,
        COUNT(*) - COUNT(sales_channel_id)     AS null_channel
    FROM raw_transactions
""").df().T.rename(columns={0: 'count'}).pipe(print)

=== ARTICLES — null counts ===
                    count
total_rows         105542
null_article_id         0
null_product_code       0
null_prod_name          0
null_colour             0
null_desc             416

=== CUSTOMERS — null counts ===
                      count
total_rows          1371980
null_customer_id          0
null_age              15861
null_fn              895050
null_active          907576
null_member_status     6062
null_news_freq        16009

=== TRANSACTIONS — null counts ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                     count
total_rows        31788324
null_date                0
null_customer_id         0
null_article_id          0
null_price               0
null_channel             0


In [52]:
# ── 4b. Value distribution checks ────────────────────────────────────────────
print("=== Sales channel distribution ===")
con.execute("""
    SELECT sales_channel_id, COUNT(*) AS tx_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM raw_transactions
    GROUP BY 1 ORDER BY 1
""").df().pipe(print)

print("\n=== Customer age distribution (bucketed) ===")
con.execute("""
    SELECT
        CASE
            WHEN age IS NULL      THEN 'Unknown'
            WHEN age < 20         THEN 'Under 20'
            WHEN age BETWEEN 20 AND 29 THEN '20-29'
            WHEN age BETWEEN 30 AND 39 THEN '30-39'
            WHEN age BETWEEN 40 AND 49 THEN '40-49'
            WHEN age BETWEEN 50 AND 59 THEN '50-59'
            ELSE '60+'
        END AS age_band,
        COUNT(*) AS customer_count
    FROM raw_customers
    GROUP BY 1 ORDER BY 2 DESC
""").df().pipe(print)

print("\n=== Top 10 product types ===")
con.execute("""
    SELECT product_type_name, COUNT(*) AS article_count
    FROM raw_articles
    GROUP BY 1 ORDER BY 2 DESC
    LIMIT 10
""").df().pipe(print)

=== Sales channel distribution ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   sales_channel_id  tx_count   pct
0                 1   9408462  29.6
1                 2  22379862  70.4

=== Customer age distribution (bucketed) ===
   age_band  customer_count
0     20-29          528358
1     30-39          234068
2     50-59          226242
3     40-49          204118
4       60+           91750
5  Under 20           71583
6   Unknown           15861

=== Top 10 product types ===
  product_type_name  article_count
0          Trousers          11169
1             Dress          10362
2           Sweater           9302
3           T-shirt           7904
4               Top           4155
5            Blouse           3979
6            Jacket           3940
7            Shorts           3939
8             Shirt           3405
9          Vest top           2991


In [53]:
# ── 5. Clean ARTICLES ─────────────────────────────────────────────────────────
# Rules:
#   - article_id: CAST to VARCHAR (Neo4j IDs must be strings)
#   - detail_desc: COALESCE null → 'No description'
#   - colour_group_name: COALESCE null → 'Unknown'
#   - perceived_colour_master_name: COALESCE null → 'Unknown'
#   - Add derived column: price_tier from grapheme_appearance_value

con.execute("""
    CREATE OR REPLACE TABLE clean_articles AS
    SELECT
        CAST(article_id AS VARCHAR)                              AS article_id,
        CAST(product_code AS VARCHAR)                            AS product_code,
        TRIM(prod_name)                                          AS prod_name,
        COALESCE(TRIM(product_type_name), 'Unknown')             AS product_type_name,
        COALESCE(TRIM(product_group_name), 'Unknown')            AS product_group_name,
        COALESCE(TRIM(graphical_appearance_name), 'Unknown')     AS graphical_appearance_name,
        COALESCE(TRIM(colour_group_name), 'Unknown')             AS colour_group_name,
        COALESCE(TRIM(perceived_colour_value_name), 'Unknown')   AS perceived_colour_value,
        COALESCE(TRIM(perceived_colour_master_name), 'Unknown')  AS perceived_colour_master,
        COALESCE(TRIM(department_name), 'Unknown')               AS department_name,
        COALESCE(TRIM(index_name), 'Unknown')                    AS index_name,
        COALESCE(TRIM(index_group_name), 'Unknown')              AS index_group_name,
        COALESCE(TRIM(section_name), 'Unknown')                  AS section_name,
        COALESCE(TRIM(garment_group_name), 'Unknown')            AS garment_group_name,
        COALESCE(TRIM(detail_desc), 'No description available')  AS detail_desc,
        -- Derived: map index_group_name to a simpler store_section label
        CASE
            WHEN index_group_name ILIKE '%ladieswear%'   THEN 'Womenswear'
            WHEN index_group_name ILIKE '%menswear%'     THEN 'Menswear'
            WHEN index_group_name ILIKE '%divided%'      THEN 'Young Fashion'
            WHEN index_group_name ILIKE '%children%'     THEN 'Kidswear'
            WHEN index_group_name ILIKE '%sport%'        THEN 'Sport'
            ELSE 'Other'
        END AS store_section
    FROM raw_articles
    WHERE article_id IS NOT NULL
""")

n = con.execute("SELECT COUNT(*) FROM clean_articles").fetchone()[0]
print(f"clean_articles: {n:,} rows")
con.execute("SELECT * FROM clean_articles LIMIT 3").df()

clean_articles: 105,542 rows


,article_id,product_code,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value,perceived_colour_master,department_name,index_name,index_group_name,section_name,garment_group_name,detail_desc,store_section
0,0108775015,0108775,Strap top,Vest top,Garment Upper body,Solid,Black,Dark,Black,Jersey Basic,Ladieswear,Ladieswear,Womens Everyday Basics,Jersey Basic,Jersey top with narrow shoulder straps.,Womenswear
1,0108775044,0108775,Strap top,Vest top,Garment Upper body,Solid,White,Light,White,Jersey Basic,Ladieswear,Ladieswear,Womens Everyday Basics,Jersey Basic,Jersey top with narrow shoulder straps.,Womenswear
2,0108775051,0108775,Strap top (1),Vest top,Garment Upper body,Stripe,Off White,Dusty Light,White,Jersey Basic,Ladieswear,Ladieswear,Womens Everyday Basics,Jersey Basic,Jersey top with narrow shoulder straps.,Womenswear


In [54]:
# ── 6. Clean CUSTOMERS ────────────────────────────────────────────────────────
# Rules:
#   - age: COALESCE null → median (29)
#   - FN / Active: COALESCE null → 0, CAST to INT
#   - club_member_status: COALESCE null → 'UNKNOWN', uppercase
#   - fashion_news_frequency: COALESCE null → 'NONE', uppercase
#   - Add derived: age_band bucket

con.execute("""
    CREATE OR REPLACE TABLE clean_customers AS
    SELECT
        TRIM(customer_id)                                        AS customer_id,
        CAST(COALESCE(FN, 0) AS INTEGER)                         AS fn_flag,
        CAST(COALESCE(Active, 0) AS INTEGER)                     AS active_flag,
        UPPER(COALESCE(TRIM(club_member_status), 'UNKNOWN'))     AS club_member_status,
        UPPER(COALESCE(TRIM(fashion_news_frequency), 'NONE'))    AS fashion_news_frequency,
        CAST(COALESCE(age, 29) AS INTEGER)                       AS age,
        TRIM(postal_code)                                        AS postal_code,
        -- Derived age band
        CASE
            WHEN COALESCE(age, 29) < 20            THEN 'Under_20'
            WHEN COALESCE(age, 29) BETWEEN 20 AND 29 THEN '20s'
            WHEN COALESCE(age, 29) BETWEEN 30 AND 39 THEN '30s'
            WHEN COALESCE(age, 29) BETWEEN 40 AND 49 THEN '40s'
            WHEN COALESCE(age, 29) BETWEEN 50 AND 59 THEN '50s'
            ELSE '60_plus'
        END AS age_band
    FROM raw_customers
    WHERE customer_id IS NOT NULL
""")

n = con.execute("SELECT COUNT(*) FROM clean_customers").fetchone()[0]
print(f"clean_customers: {n:,} rows")
con.execute("SELECT * FROM clean_customers LIMIT 3").df()

clean_customers: 1,371,980 rows


,customer_id,fn_flag,active_flag,club_member_status,fashion_news_frequency,age,postal_code,age_band
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,40s
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,20s
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,20s


In [55]:
# ── 7. Clean TRANSACTIONS ─────────────────────────────────────────────────────
# Rules:
#   - t_dat: CAST to DATE
#   - article_id: CAST to VARCHAR (match node ID format)
#   - price: ROUND to 4 dp (H&M stores raw price with many decimals)
#   - sales_channel_id: CAST to INT, map to label
#   - Filter: only keep transactions where both customer and article exist in clean tables
#   - Add derived: year_month for temporal analysis

con.execute("""
    CREATE OR REPLACE TABLE clean_transactions AS
    SELECT
        CAST(t.t_dat AS DATE)                            AS tx_date,
        STRFTIME(CAST(t.t_dat AS DATE), '%Y-%m')         AS year_month,
        TRIM(t.customer_id)                              AS customer_id,
        CAST(t.article_id AS VARCHAR)                    AS article_id,
        ROUND(CAST(t.price AS DOUBLE), 4)                AS price,
        CAST(t.sales_channel_id AS INTEGER)              AS sales_channel_id,
        CASE CAST(t.sales_channel_id AS INTEGER)
            WHEN 1 THEN 'In_Store'
            WHEN 2 THEN 'Online'
            ELSE 'Unknown'
        END                                              AS channel_label
    FROM raw_transactions t
    -- Only keep rows where both customer and article are valid (inner join filter)
    INNER JOIN clean_customers c  ON TRIM(t.customer_id) = c.customer_id
    INNER JOIN clean_articles  a  ON CAST(t.article_id AS VARCHAR) = a.article_id
    WHERE t.t_dat IS NOT NULL
      AND t.price  IS NOT NULL
      AND t.price  > 0
""")

n = con.execute("SELECT COUNT(*) FROM clean_transactions").fetchone()[0]
print(f"clean_transactions: {n:,} rows")
con.execute("SELECT * FROM clean_transactions LIMIT 5").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

clean_transactions: 31,788,324 rows


,tx_date,year_month,customer_id,article_id,price,sales_channel_id,channel_label
0,2018-09-28,2018-09,3d3671f6a2c3ebe6c2fcedfb3033846943131a001d3481...,0662868003,0.0339,1,In_Store
1,2018-09-28,2018-09,3d378063ba775a0c20fdf10f907621a57ded5e45c979e9...,0517931006,0.0508,1,In_Store
2,2018-09-28,2018-09,3d378063ba775a0c20fdf10f907621a57ded5e45c979e9...,0637661004,0.0508,1,In_Store
3,2018-09-28,2018-09,3d378063ba775a0c20fdf10f907621a57ded5e45c979e9...,0573716040,0.0254,2,Online
4,2018-09-28,2018-09,3d378063ba775a0c20fdf10f907621a57ded5e45c979e9...,0573085020,0.0339,2,Online


In [56]:
# ── 8. Quick sanity checks ────────────────────────────────────────────────────
print("=== Price range in clean transactions ===")
con.execute("""
    SELECT
        ROUND(MIN(price),4) AS min_price,
        ROUND(MAX(price),4) AS max_price,
        ROUND(AVG(price),4) AS avg_price,
        ROUND(MEDIAN(price),4) AS median_price
    FROM clean_transactions
""").df().pipe(print)

print("\n=== Transactions per channel ===")
con.execute("""
    SELECT channel_label, COUNT(*) AS tx_count,
           ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),2) AS pct
    FROM clean_transactions
    GROUP BY 1 ORDER BY 2 DESC
""").df().pipe(print)

print("\n=== Transactions per year-month (first 6) ===")
con.execute("""
    SELECT year_month, COUNT(*) AS tx_count
    FROM clean_transactions
    GROUP BY 1 ORDER BY 1
    LIMIT 6
""").df().pipe(print)

=== Price range in clean transactions ===
   min_price  max_price  avg_price  median_price
0        0.0     0.5915     0.0278        0.0254

=== Transactions per channel ===
  channel_label  tx_count   pct
0        Online  22379862  70.4
1      In_Store   9408462  29.6

=== Transactions per year-month (first 6) ===
  year_month  tx_count
0    2018-09    594776
1    2018-10   1397040
2    2018-11   1270619
3    2018-12   1148827
4    2019-01   1263471
5    2019-02   1152412


In [57]:
# ── 9. Export to PARQUET (analytical side) ────────────────────────────────────
# Parquet is columnar — fast for aggregations in later sessions

parquet_exports = [
    ('clean_articles',     'C:/Users/prasa/Downloads/H&M/parquet/articles.parquet'),
    ('clean_customers',    'C:/Users/prasa/Downloads/H&M/parquet/customers.parquet'),
    ('clean_transactions', 'C:/Users/prasa/Downloads/H&M/parquet/transactions.parquet'),
]

for table, path in parquet_exports:
    con.execute(f"""
        COPY {table} TO '{path}'
        (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    size_mb = Path(path).stat().st_size / 1e6
    print(f"  ✅  {path:45s}  {size_mb:.1f} MB")

  ✅  C:/Users/prasa/Downloads/H&M/parquet/articles.parquet  5.7 MB
  ✅  C:/Users/prasa/Downloads/H&M/parquet/customers.parquet  168.3 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅  C:/Users/prasa/Downloads/H&M/parquet/transactions.parquet  808.0 MB


In [58]:
# ── 10. Export Neo4j-ready CSVs ───────────────────────────────────────────────

# ── NODE: Article ─────────────────────────────────────────────────────────────
con.execute("""
    COPY (
        SELECT
            article_id          AS articleId,
            prod_name           AS name,
            product_type_name   AS productType,
            product_group_name  AS productGroup,
            colour_group_name   AS colourGroup,
            perceived_colour_master AS colour,
            department_name     AS department,
            index_group_name    AS indexGroup,
            section_name        AS section,
            garment_group_name  AS garmentGroup,
            store_section       AS storeSection,
            detail_desc         AS description
        FROM clean_articles
    ) TO 'C:/Users/prasa/Downloads/H&M/neo4j/neo4j_node_Article.csv'
    (FORMAT CSV, HEADER true)
""")
n = con.execute("SELECT COUNT(*) FROM clean_articles").fetchone()[0]
print(f"  ✅  neo4j_node_Article.csv          {n:>10,} rows")

# ── NODE: Customer ────────────────────────────────────────────────────────────
con.execute("""
    COPY (
        SELECT
            customer_id            AS customerId,
            age                    AS age,
            age_band               AS ageBand,
            club_member_status     AS memberStatus,
            fashion_news_frequency AS newsFrequency,
            fn_flag                AS fnFlag,
            active_flag            AS activeFlag,
            postal_code            AS postalCode
        FROM clean_customers
    ) TO 'C:/Users/prasa/Downloads/H&M/neo4j/neo4j_node_Customer.csv'
    (FORMAT CSV, HEADER true)
""")
n = con.execute("SELECT COUNT(*) FROM clean_customers").fetchone()[0]
print(f"  ✅  neo4j_node_Customer.csv         {n:>10,} rows")

# ── RELATIONSHIP: PURCHASED ───────────────────────────────────────────────────
con.execute("""
    COPY (
        SELECT
            customer_id   AS customerId,
            article_id    AS articleId,
            tx_date       AS txDate,
            year_month    AS yearMonth,
            price         AS price,
            channel_label AS channel
        FROM clean_transactions
    ) TO 'C:/Users/prasa/Downloads/H&M/neo4j/neo4j_rel_PURCHASED.csv'
    (FORMAT CSV, HEADER true)
""")
n = con.execute("SELECT COUNT(*) FROM clean_transactions").fetchone()[0]
print(f"  ✅  neo4j_rel_PURCHASED.csv         {n:>10,} rows")

  ✅  neo4j_node_Article.csv             105,542 rows
  ✅  neo4j_node_Customer.csv          1,371,980 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅  neo4j_rel_PURCHASED.csv         31,788,324 rows


In [59]:
# ── 11. Preview exported CSV headers ─────────────────────────────────────────
import csv

for fname in ['neo4j_node_Article.csv',
              'neo4j_node_Customer.csv',
              'neo4j_rel_PURCHASED.csv']:
    path = Path('C:/Users/prasa/Downloads/H&M/neo4j') / fname
    with open(path) as f:
        reader = csv.reader(f)
        headers = next(reader)
        first   = next(reader)
    print(f"\n{fname}")
    print(f"  Columns : {headers}")
    print(f"  Row 1   : {first}")


neo4j_node_Article.csv
  Columns : ['articleId', 'name', 'productType', 'productGroup', 'colourGroup', 'colour', 'department', 'indexGroup', 'section', 'garmentGroup', 'storeSection', 'description']
  Row 1   : ['0108775015', 'Strap top', 'Vest top', 'Garment Upper body', 'Black', 'Black', 'Jersey Basic', 'Ladieswear', 'Womens Everyday Basics', 'Jersey Basic', 'Womenswear', 'Jersey top with narrow shoulder straps.']

neo4j_node_Customer.csv
  Columns : ['customerId', 'age', 'ageBand', 'memberStatus', 'newsFrequency', 'fnFlag', 'activeFlag', 'postalCode']
  Row 1   : ['00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657', '49', '40s', 'ACTIVE', 'NONE', '0', '0', '52043ee2162cf5aa7ee79974281641c6f11a68d276429a91f8ca0d4b6efa8100']

neo4j_rel_PURCHASED.csv
  Columns : ['customerId', 'articleId', 'txDate', 'yearMonth', 'price', 'channel']
  Row 1   : ['3d3671f6a2c3ebe6c2fcedfb3033846943131a001d34817043818d87bea3fba6', '0662868003', '2018-09-28', '2018-09', '0.0339', 'In_Store'

In [60]:
# ── 12. Summary ───────────────────────────────────────────────────────────────
print("=" * 60)
print("  ETL COMPLETE — Session 2 Deliverable")
print("=" * 60)

summary = [
    ('Parquet',   'C:/Users/prasa/Downloads/H&M/parquet/articles.parquet'),
    ('Parquet',   'C:/Users/prasa/Downloads/H&M/parquet/customers.parquet'),
    ('Parquet',   'C:/Users/prasa/Downloads/H&M/parquet/transactions.parquet'),
    ('Neo4j CSV', 'C:/Users/prasa/Downloads/H&M/neo4j/neo4j_node_Article.csv'),
    ('Neo4j CSV', 'C:/Users/prasa/Downloads/H&M/neo4j/neo4j_node_Customer.csv'),
    ('Neo4j CSV', 'C:/Users/prasa/Downloads/H&M/neo4j/neo4j_rel_PURCHASED.csv'),
]

for kind, path in summary:
    p = Path(path)
    size = p.stat().st_size / 1e6 if p.exists() else 0
    print(f"  {kind:<12}  {path:<65}  {size:>8.1f} MB")

print()
print("  Next: 02_graph_load.ipynb — LOAD CSV into Neo4j")
con.close()

  ETL COMPLETE — Session 2 Deliverable
  Parquet       C:/Users/prasa/Downloads/H&M/parquet/articles.parquet                   5.7 MB
  Parquet       C:/Users/prasa/Downloads/H&M/parquet/customers.parquet                168.3 MB
  Parquet       C:/Users/prasa/Downloads/H&M/parquet/transactions.parquet             808.0 MB
  Neo4j CSV     C:/Users/prasa/Downloads/H&M/neo4j/neo4j_node_Article.csv              29.2 MB
  Neo4j CSV     C:/Users/prasa/Downloads/H&M/neo4j/neo4j_node_Customer.csv            213.4 MB
  Neo4j CSV     C:/Users/prasa/Downloads/H&M/neo4j/neo4j_rel_PURCHASED.csv           3481.8 MB

  Next: 02_graph_load.ipynb — LOAD CSV into Neo4j
